In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

df_full=pd.read_csv("../data/RELIANCE_features_v2.csv", index_col=0, parse_dates=True)

X_Test=pd.read_csv("../data/X_test.csv", index_col=0, parse_dates=True)
Y_Test=pd.read_csv("../data/Y_test.csv", index_col=0, parse_dates=True).squeeze() #.squeeze convert the Date and target row coloumn format to a series format where Data and target are on entry only



from xgboost import XGBClassifier

import pickle
with open("../data/model.pkl", "rb") as f:
    model=pickle.load(f)

print("MODEL LOADED")
print(f"Test Period: {X_Test.index[0]} to {X_Test.index[-1]}")






MODEL LOADED
Test Period: 2023-01-11 00:00:00 to 2023-12-29 00:00:00


In [19]:
y_pred = model.predict(X_Test)
y_prob = model.predict_proba(X_Test)[:, 1] #gets the probability of getting 1

#build a results dataframe
results=pd.DataFrame(index=X_Test.index)
results["Actual"]=Y_Test.values
results["Predicted"]=y_pred
results["Probability"]=y_prob
results["Close"]= df_full.loc[X_Test.index, "Close"]

print(results.head(20))
print(f"Predicted UP: {(results['Predicted']==1).sum()}")
print(f"Predicted DOWN: {(results['Predicted']==0).sum()}")




            Actual  Predicted  Probability        Close
2023-01-11       0          0     0.406797  1153.179077
2023-01-12       1          0     0.297726  1128.276978
2023-01-13       0          0     0.359467  1126.451172
2023-01-16       0          0     0.410935  1115.723389
2023-01-17       0          0     0.386854  1131.563965
2023-01-18       0          0     0.343193  1129.692139
2023-01-19       0          0     0.345707  1128.482544
2023-01-20       0          0     0.425608  1115.061523
2023-01-23       0          0     0.421591  1109.423706
2023-01-24       0          0     0.385046  1102.873047
2023-01-25       0          1     0.644989  1087.625977
2023-01-27       0          1     0.744947  1066.992432
2023-01-30       0          1     0.721075  1077.218018
2023-01-31       0          1     0.684755  1074.524414
2023-02-01       1          1     0.695097  1068.156372
2023-02-02       1          1     0.705838  1062.244751
2023-02-03       1          1     0.731628  1063

What does predict_proba() give you that predict() doesn't? It gives me the probability of which the model thinks that the output should be 1
What does [:, 1] mean in y_prob = model.predict_proba(X_test)[:, 1]? Means the probability of getting a 1 according to model 

In [ ]:
from IPython.display import display



def backtest_stratergy(results, initial_capital=100000, transaction_cost=0.001, threshold=0.5):
    """We are simply doing these steps: 
        -If currently not holding and model predicts UP, we buy
        -If holding and model predicts DOWN, we sell
        -A simple transaction cost on every share bought and sold
        -We have kept the proability threshold to be .5 here
    
    """

    capital= initial_capital
    position=0 # number of shares held
    portfolio=[] #closing value of portfolio each day
    cash=[]
    stock_value=[]
    buy=True    #track if we can buy right now or not


    for i in range(len(results)):

        price=results["Close"].iloc[i]
        prob=results["Probability"].iloc[i]

        if (buy and prob>=threshold):
            shares_bought=capital/(price*(1+transaction_cost))
            capital=0
            position=shares_bought
            buy=False

        elif (not buy and prob<threshold):
            money_got=position*(price-transaction_cost)
            capital=money_got
            position=0
            buy=True

        total_value=position*price+capital
        portfolio.append(total_value)
        cash.append(capital)
        stock_value.append(position*price)
    results["CashInHand"]=cash
    results["Stock_Value"]=stock_value
    results["Stratergy_Value"]= portfolio
    return results


backtest_stratergy(results, 100000, 0.001, 0.5)

#pd.set_option('display.max_rows', None)
display(results)






What does transaction_cost=0.001 mean? What percentage is that? 0.1% of share cost
Why do we track in_trade as a boolean? 
What happens to our capital when we're not in a trade?
Why do we subtract transaction cost on both buy and sell?